# Module 02 — Lesson 07
# Exception Handling

---

> A data pipeline that crashes on the first dirty row is not a pipeline — it's a prototype.

Every function in your data toolkit can fail on bad input:
- `int("abc")` → `ValueError`
- `row["email"]` on a row with no email → `KeyError`
- `total / count` when count is zero → `ZeroDivisionError`
- `None.strip()` on a missing name → `AttributeError`

In a toy notebook these crashes are fine — you fix them and re-run. But in a real pipeline processing 100,000 records, you cannot stop at the first bad row. You need to:
1. Catch the error
2. Log what went wrong and which row caused it
3. Continue with the next record
4. Deliver both the clean results and a full error report at the end

That is what exception handling enables. It is the difference between a script that works on your test data and a system that works in production.

---
## 🎯 Learning Objectives

By the end of this lesson, you will be able to:

- Identify Python's most common exception types and when each is raised
- Use `try` / `except` to catch and handle errors gracefully
- Catch specific exception types, multiple types, and use `as e` to inspect the error
- Use `else` (runs on success) and `finally` (always runs) correctly
- Raise exceptions deliberately with `raise` to signal invalid inputs
- Define lightweight custom exception classes
- Apply the three core patterns: safe converter, batch processor with error log, resilient pipeline

---
## 1. What Is an Exception?

An **exception** is Python's way of signalling that something went wrong at runtime. When Python encounters an operation it cannot perform, it **raises** an exception. If you don't handle it, Python prints a traceback and stops.

There are two kinds of errors:
- **Syntax errors** — caught before the code runs (wrong indentation, missing colon)
- **Runtime exceptions** — raised while the code is running (bad data, wrong type, missing key)

In [ ]:
# The seven exceptions you'll meet most in data science

demos = [
    ("ValueError",       lambda: int("abc")),
    ("TypeError",        lambda: "score" + 88),
    ("KeyError",         lambda: {}["name"]),
    ("IndexError",       lambda: [][0]),
    ("ZeroDivisionError",lambda: 1 / 0),
    ("AttributeError",   lambda: None.strip()),
    ("NameError",        lambda: undefined_variable),
]

for exc_name, fn in demos:
    try:
        fn()
    except Exception as e:
        print(f"  {exc_name:<22} → {e}")

| Exception | Raised when |
|---|---|
| `ValueError` | Right type, wrong value — `int("abc")`, `float("N/A")` |
| `TypeError` | Wrong type entirely — `"text" + 5`, `None + 1` |
| `KeyError` | Dict key doesn't exist — `d["missing_key"]` |
| `IndexError` | List index out of range — `[][0]` |
| `ZeroDivisionError` | Division by zero — `x / 0` |
| `AttributeError` | Method/attribute doesn't exist — `None.strip()` |
| `FileNotFoundError` | File path doesn't exist — `open("nope.csv")` |

---
## 2. `try` / `except` — Catching Exceptions

```python
try:
    # code that might fail
    risky_operation()
except SomeException:
    # runs only if SomeException was raised
    handle_the_problem()
```

The `try` block runs normally. If an exception is raised, Python jumps immediately to the matching `except` block. Code after the failing line in `try` is skipped.

In [ ]:
# Without exception handling — crash
def divide_unsafe(a, b):
    return a / b

print(divide_unsafe(10, 2))    # 5.0
# divide_unsafe(10, 0)         # ZeroDivisionError — would crash here

# With exception handling — graceful
def divide_safe(a, b):
    try:
        return a / b
    except ZeroDivisionError:
        return None             # caller gets None, not a crash

print(divide_safe(10, 2))      # 5.0
print(divide_safe(10, 0))      # None — no crash

In [ ]:
# Use 'as e' to inspect the exception message
test_values = ["42", "3.14", "abc", None, "", "99"]

for val in test_values:
    try:
        result = int(float(val))   # convert string → float → int
        print(f"  {str(val)!r:<8} → {result}")
    except (ValueError, TypeError) as e:
        print(f"  {str(val)!r:<8} → Failed: {e}")

### 2.1 Catching Specific vs Multiple Exceptions

Always be as specific as possible. Catching the wrong exception type can hide real bugs.

In [ ]:
# Catching a single specific type
def to_int(value):
    try:
        return int(value)
    except ValueError:
        return None

print(to_int("42"), to_int("abc"), to_int("3.7"))

In [ ]:
# Catching multiple types in one except — use a tuple
def safe_convert(value, target_type):
    """Convert value to target_type, returning None on any failure."""
    try:
        return target_type(value)
    except (ValueError, TypeError):    # tuple of exception types
        return None

print(safe_convert("42",  int))    # 42
print(safe_convert("abc", int))    # None — ValueError
print(safe_convert(None,  float))  # None — TypeError
print(safe_convert("3.5", float))  # 3.5

In [ ]:
# Multiple separate except blocks — different handling per error type
def process_field(record, field):
    """Extract and convert a numeric field from a record."""
    try:
        raw    = record[field]       # may raise KeyError
        value  = float(raw)          # may raise ValueError or TypeError
        result = value / 100         # may raise ZeroDivisionError (not here)
        return result
    except KeyError:
        print(f"  Field '{field}' missing from record")
        return None
    except (ValueError, TypeError) as e:
        print(f"  Cannot convert '{field}' value {record.get(field)!r}: {e}")
        return None

records = [
    {"name": "Priya",  "score": "88"},
    {"name": "Rohan"},                  # missing 'score'
    {"name": "Meera",  "score": "N/A"}, # bad value
    {"name": "Karan",  "score": None},  # None value
]

for r in records:
    result = process_field(r, "score")
    if result is not None:
        print(f"  {r['name']}: score/100 = {result:.2f}")

---
## 3. `else` and `finally`

```python
try:
    risky()
except SomeError:
    handle_error()
else:
    # runs ONLY if no exception was raised in try
    on_success()
finally:
    # ALWAYS runs — exception or not
    cleanup()
```

In [ ]:
# else: runs only when try succeeded — keeps success logic separate from the try block

def parse_and_classify(raw_score):
    """Parse a score string and classify it."""
    try:
        score = float(raw_score)
    except (ValueError, TypeError) as e:
        print(f"  Parse failed for {raw_score!r}: {e}")
        return None
    else:
        # Only reaches here if float() succeeded
        grade = "A" if score >= 80 else "B" if score >= 60 else "C" if score >= 40 else "F"
        print(f"  Parsed {raw_score!r} → {score} → Grade {grade}")
        return grade

inputs = ["88", "72.5", "N/A", "", "95", None]
for inp in inputs:
    parse_and_classify(inp)

In [ ]:
# finally: always runs — use for cleanup (closing files, DB connections, etc.)

def load_data_simulation(source, should_fail=False):
    """Simulate loading data from a source."""
    connection_open = False
    try:
        print(f"  [try]     Opening connection to '{source}'")
        connection_open = True

        if should_fail:
            raise ValueError("Source returned malformed data")

        data = [1, 2, 3, 4, 5]   # simulated data
        print(f"  [try]     Loaded {len(data)} records")
        return data

    except ValueError as e:
        print(f"  [except]  Error loading data: {e}")
        return []

    finally:
        # This ALWAYS runs — connection must always be closed
        if connection_open:
            print(f"  [finally] Closing connection (always happens)")

print("Case 1 — Success:")
result = load_data_simulation("database.csv", should_fail=False)
print(f"  Result: {result}")

print("\nCase 2 — Failure:")
result = load_data_simulation("corrupt.csv", should_fail=True)
print(f"  Result: {result}")

---
## 4. Raising Exceptions

You can raise exceptions yourself to signal that a caller passed invalid input. This makes your functions behave like Python's built-ins — they fail loudly and early with a clear message, rather than silently producing wrong output.

In [ ]:
# raise stops execution immediately, like Python's built-in errors
def compute_mean(values):
    """Return mean of a non-empty list of numbers."""
    if not isinstance(values, list):
        raise TypeError(f"Expected a list, got {type(values).__name__}")
    if len(values) == 0:
        raise ValueError("Cannot compute mean of an empty list")
    return sum(values) / len(values)

# Normal use
print(compute_mean([85, 72, 91]))   # 82.666...

# Bad inputs — raise clear errors
try:
    compute_mean("not a list")
except TypeError as e:
    print(f"TypeError: {e}")

try:
    compute_mean([])
except ValueError as e:
    print(f"ValueError: {e}")

In [ ]:
# Raise in a data validation function — fail fast with a clear message

def validate_score(score, field_name="score"):
    """
    Validate that score is a number in [0, 100].
    Returns the validated float value.
    Raises ValueError if invalid.
    """
    if score is None:
        raise ValueError(f"{field_name}: value is None (required)")
    try:
        score = float(score)
    except (ValueError, TypeError):
        raise ValueError(f"{field_name}: cannot convert {score!r} to number")
    if not (0 <= score <= 100):
        raise ValueError(f"{field_name}: {score} is out of valid range [0, 100]")
    return score


test_cases = [88, "72.5", None, "abc", 115, 0, "100"]
for val in test_cases:
    try:
        result = validate_score(val)
        print(f"  {str(val)!r:<10} → valid: {result}")
    except ValueError as e:
        print(f"  {str(val)!r:<10} → ✗ {e}")

In [ ]:
# Re-raising: catch, log, then re-raise for the caller to handle

import traceback

def process_record(record):
    try:
        score = validate_score(record.get("score"))
        return {**record, "score": score, "valid": True}
    except ValueError as e:
        # Log the problem, then re-raise so caller knows it failed
        print(f"  [LOG] Validation failed for {record.get('name')!r}: {e}")
        raise   # bare raise — re-raises the exact same exception

try:
    process_record({"name": "Priya", "score": "abc"})
except ValueError:
    print("  [CALLER] Caught the re-raised exception — can decide what to do")

---
## 5. Custom Exception Classes

When you build a data toolkit or library, standard exceptions (`ValueError`, `TypeError`) are too generic for callers to distinguish between errors. Custom exceptions let callers catch *your* specific errors separately.

In [ ]:
# Define a hierarchy of custom exceptions for a data pipeline

class DataError(Exception):
    """Base class for all data pipeline errors."""
    pass

class MissingFieldError(DataError):
    """A required field is absent from the record."""
    def __init__(self, field, record_id=None):
        self.field     = field
        self.record_id = record_id
        super().__init__(
            f"Required field '{field}' is missing"
            + (f" (record id={record_id})" if record_id else "")
        )

class InvalidValueError(DataError):
    """A field contains a value that fails validation."""
    def __init__(self, field, value, reason):
        self.field  = field
        self.value  = value
        self.reason = reason
        super().__init__(f"Field '{field}' has invalid value {value!r}: {reason}")


# Raise and catch custom exceptions
def validate_record(record):
    if "name" not in record or not record["name"]:
        raise MissingFieldError("name", record_id=record.get("id"))

    score = record.get("score")
    if score is None:
        raise MissingFieldError("score", record_id=record.get("id"))
    if not isinstance(score, (int, float)) or not (0 <= score <= 100):
        raise InvalidValueError("score", score, "must be a number in [0, 100]")

    return True


test_records = [
    {"id": 1, "name": "Priya",  "score": 88},
    {"id": 2, "name": "",       "score": 72},    # missing name
    {"id": 3, "name": "Meera",  "score": None},  # missing score
    {"id": 4, "name": "Karan",  "score": 115},   # invalid score
]

for rec in test_records:
    try:
        validate_record(rec)
        print(f"  ID {rec['id']}: ✓ OK")
    except MissingFieldError as e:
        print(f"  ID {rec['id']}: ✗ MissingField  — {e}")
    except InvalidValueError as e:
        print(f"  ID {rec['id']}: ✗ InvalidValue  — {e}")
    except DataError as e:
        print(f"  ID {rec['id']}: ✗ DataError     — {e}")

---
## 6. The Exception Hierarchy

All built-in exceptions inherit from `BaseException`. In practice, you almost always catch `Exception` or a specific subclass — never catch `BaseException` (it includes `SystemExit` and `KeyboardInterrupt`, which should not be suppressed).

In [ ]:
# The hierarchy you need to know
hierarchy = """
BaseException
├── SystemExit           ← raised by sys.exit()  — don't catch
├── KeyboardInterrupt    ← Ctrl+C                — don't catch
└── Exception            ← all normal errors (catch this or subclasses)
    ├── ValueError
    ├── TypeError
    ├── KeyError         ←┐
    ├── IndexError       ←┤ all subclass LookupError
    ├── AttributeError
    ├── ZeroDivisionError←┐
    ├── OverflowError    ←┤ all subclass ArithmeticError
    ├── FileNotFoundError←┐
    ├── PermissionError  ←┤ all subclass OSError
    └── RuntimeError
"""
print(hierarchy)

# Because of inheritance, catching a parent catches all children
try:
    {}["missing_key"]       # KeyError — subclass of LookupError
except LookupError as e:
    print(f"Caught as LookupError: {type(e).__name__}: {e}")

---
## 7. Applied — Three Core Data Science Patterns

### 7.1 — Safe Converter

Convert a value to a target type, returning a default instead of crashing. This pattern replaces every `if isinstance(...)` chain you'd otherwise write.

In [ ]:
def safe_cast(value, target_type, default=None):
    """
    Convert value to target_type.
    Returns default if conversion fails for any reason.
    Works for int, float, str, bool.
    """
    if value is None:
        return default
    try:
        if target_type == int:
            return int(float(value))   # handles "42.0" → 42
        return target_type(value)
    except (ValueError, TypeError):
        return default


# Comprehensive test table
cases = [
    ("42",     int,   None, 42),
    ("42.9",   int,   None, 42),       # truncated via float
    ("3.14",   float, None, 3.14),
    (None,     float, 0.0,  0.0),
    ("",       int,   -1,   -1),
    ("abc",    float, None, None),
    (True,     int,   None, 1),        # True → 1
    ("True",   str,   None, "True"),
]

print(f"  {'Input':<10} {'Type':<7} {'Default':<9} {'Got':<8} {'Pass?'}")
print("  " + "─" * 46)
for val, typ, default, expected in cases:
    result = safe_cast(val, typ, default)
    passed = result == expected
    print(f"  {str(val)!r:<10} {typ.__name__:<7} {str(default):<9} "
          f"{str(result):<8} {'✓' if passed else '✗'}")

### 7.2 — Batch Processor with Error Log

Process every record in a list. On failure: log the error and continue. Never abort the whole batch because of one bad row.

In [ ]:
def process_student_record(raw):
    """
    Clean and validate a raw student record dict.
    Raises ValueError describing what's wrong.
    Returns a clean dict on success.
    """
    name = str(raw.get("name") or "").strip().title()
    if not name:
        raise ValueError("name is empty or missing")

    age = safe_cast(raw.get("age"), int)
    if age is None or not (15 <= age <= 35):
        raise ValueError(f"age {raw.get('age')!r} is invalid (expected 15–35)")

    score = safe_cast(raw.get("score"), float)
    if score is None or not (0 <= score <= 100):
        raise ValueError(f"score {raw.get('score')!r} is invalid (expected 0–100)")

    return {"name": name, "age": age, "score": score,
            "grade": "A" if score >= 80 else "B" if score >= 60 else "C" if score >= 40 else "F"}


def process_batch(raw_records):
    """
    Process a list of raw records.
    Returns (clean_records, error_log).
    """
    clean_records = []
    error_log     = []

    for i, raw in enumerate(raw_records):
        try:
            clean = process_student_record(raw)
            clean_records.append(clean)
        except ValueError as e:
            error_log.append({
                "row"    : i + 1,
                "raw"    : raw,
                "error"  : str(e),
            })

    return clean_records, error_log


# A realistic dataset with dirty rows
raw_batch = [
    {"name": "  PRIYA SHARMA ", "age": "20",  "score": "88"},
    {"name": "rohan verma",     "age": "22",  "score": "72.5"},
    {"name": "",                "age": "19",  "score": "95"},   # empty name
    {"name": "karan mehta",     "age": "abc", "score": "61"},   # bad age
    {"name": "ananya singh",    "age": "22",  "score": "110"},  # score > 100
    {"name": "dev patel",       "age": "25",  "score": "77"},
    {"name": "riya joshi",      "age": "99",  "score": "82"},   # age out of range
    {"name": "arjun nair",      "age": "21",  "score": None},   # missing score
    {"name": "divya rao",       "age": "23",  "score": "68"},
]

clean, errors = process_batch(raw_batch)

print(f"Batch result: {len(clean)} clean / {len(errors)} errors / {len(raw_batch)} total\n")

print("Clean records:")
print(f"  {'Name':<16} {'Age':>4} {'Score':>6} {'Grade'}")
print("  " + "─" * 35)
for r in clean:
    print(f"  {r['name']:<16} {r['age']:>4} {r['score']:>6.1f} {r['grade']}")

print("\nError log:")
print(f"  {'Row':>4}  {'Error'}")
print("  " + "─" * 50)
for err in errors:
    print(f"  Row {err['row']:>2}: {err['error']}")

### 7.3 — Resilient Pipeline with Summary Report

In [ ]:
def run_pipeline(raw_records, processor_fn):
    """
    Generic pipeline: apply processor_fn to every record.
    Returns a summary dict with results, errors, and statistics.
    """
    results   = []
    error_log = []

    for i, raw in enumerate(raw_records):
        try:
            result = processor_fn(raw)
            results.append(result)
        except Exception as e:
            error_log.append({
                "row"       : i + 1,
                "error_type": type(e).__name__,
                "message"   : str(e),
                "raw"       : raw,
            })

    total  = len(raw_records)
    n_ok   = len(results)
    n_err  = len(error_log)

    # Error breakdown by type
    err_types = {}
    for e in error_log:
        err_types[e["error_type"]] = err_types.get(e["error_type"], 0) + 1

    return {
        "results"     : results,
        "errors"      : error_log,
        "total"       : total,
        "success"     : n_ok,
        "failed"      : n_err,
        "success_rate": round(n_ok / total * 100, 1) if total else 0,
        "error_types" : err_types,
    }


summary = run_pipeline(raw_batch, process_student_record)

print("Pipeline Summary")
print("═" * 40)
print(f"  Total records    : {summary['total']}")
print(f"  Successfully processed : {summary['success']}")
print(f"  Failed           : {summary['failed']}")
print(f"  Success rate     : {summary['success_rate']}%")
print(f"  Error breakdown  : {summary['error_types']}")

if summary["results"]:
    scores = [r["score"] for r in summary["results"]]
    print(f"\n  Score stats on clean data:")
    print(f"    Mean : {sum(scores)/len(scores):.1f}")
    print(f"    Min  : {min(scores)}")
    print(f"    Max  : {max(scores)}")

---
## ⚠️ Common Mistakes

In [ ]:
# ── Mistake 1: Catching too broadly — hides real bugs ────────────────────────

# ❌ Catches EVERYTHING — even bugs in your own code you should fix
def bad_process(data):
    try:
        result = data["score"] * 2
        return reslt    # typo: 'reslt' — NameError should crash!
    except Exception:   # swallows the NameError silently
        return None

print(f"Bad (hides bug): {bad_process({'score': 5})}")
# Returns None — you never see the NameError typo

# ✅ Catch only the errors you expect and know how to handle
def good_process(data):
    try:
        return data["score"] * 2
    except KeyError:
        return None       # only catch the 'score might be missing' case

print(f"Good (specific): {good_process({'score': 5})}")

In [ ]:
# ── Mistake 2: Silent except — the worst anti-pattern ────────────────────────

# ❌ Never do this — errors vanish completely
def silent_fail(value):
    try:
        return int(value)
    except:
        pass    # error swallowed — caller has no idea what happened

result = silent_fail("abc")
print(f"Silent fail returned: {result!r}")   # None — but why? caller doesn't know

# ✅ At minimum, log the error; better, return a meaningful default + reason
def transparent_fail(value):
    try:
        return int(value), None
    except (ValueError, TypeError) as e:
        return None, str(e)    # return both result and error reason

val, err = transparent_fail("abc")
print(f"Transparent fail: value={val!r}, error={err!r}")

In [ ]:
# ── Mistake 3: Using exceptions for normal control flow ──────────────────────

scores = [85, 72, 91, 60, 38]

# ❌ Using try/except as an if statement — exceptions are slow
first_passing = None
for s in scores:
    try:
        if s < 40:
            raise StopIteration    # misusing exception for flow
        first_passing = s
        break
    except StopIteration:
        pass

# ✅ Use normal conditions for expected/normal decisions
first_passing = next((s for s in scores if s >= 40), None)
print(f"First passing score: {first_passing}")

# Exception handling is for EXCEPTIONAL / UNEXPECTED conditions
# Normal branching should use if/else and loops

In [ ]:
# ── Mistake 4: except clause order — most specific must come first ────────────

# ❌ Wrong order — ValueError is a subclass of Exception,
#    so the broad Exception catch runs first, ValueError block never reached
def wrong_order(value):
    try:
        return int(value)
    except Exception as e:      # catches everything, including ValueError
        return f"Generic: {e}"
    except ValueError as e:     # unreachable!
        return f"ValueError: {e}"

# ✅ Specific exceptions first, broad ones last
def right_order(value):
    try:
        return int(value)
    except ValueError as e:     # specific first
        return f"ValueError: {e}"
    except Exception as e:      # broad last (catch-all fallback)
        return f"Unexpected: {e}"

print(f"Wrong order: {wrong_order('abc')}")
print(f"Right order: {right_order('abc')}")

---
## ✏️ Practice Exercises

### Exercise 1 — Starter: Safe Division

Write `safe_divide(a, b)` that:
- Returns `a / b` normally
- Returns `None` if `b == 0` (with a print message)
- Returns `None` if either argument is not a number (with a print message)
- Uses specific exception types, not a bare `except`

In [ ]:
def safe_divide(a, b):
    # Your code here
    pass

# Tests
cases = [(10, 2), (10, 0), ("10", 2), (10, None), (0, 5), (-6, 3)]
for a, b in cases:
    result = safe_divide(a, b)
    print(f"  safe_divide({a!r:>5}, {str(b)!r:>5}) → {result!r}")

### Exercise 2 — Robust CSV Row Parser

A CSV row arrives as a plain list of strings. Write `parse_csv_row(row, schema)` that converts each field according to a schema dict, collecting all conversion errors instead of aborting at the first one.

```python
schema = {
    "name" : str,
    "age"  : int,
    "score": float,
}
```

Return `(parsed_dict, error_list)`. The error list should be empty on a clean row.

In [ ]:
def parse_csv_row(field_names, raw_values, schema):
    """
    Parse a CSV row (list of strings) into typed values using schema.
    Returns (parsed_dict, error_list).
    Collects ALL errors — does not abort on first failure.
    """
    # Your code here
    pass


SCHEMA = {"name": str, "age": int, "score": float}
FIELDS = ["name", "age", "score"]

rows = [
    ["Priya", "20",  "88.5"],   # clean
    ["Rohan", "abc", "72.0"],   # bad age
    ["Meera", "19",  "N/A"],    # bad score
    ["Karan", "xyz", "zzz"],    # two bad fields
    ["Dev",   "25",  "77"],     # clean
]

for raw in rows:
    parsed, errors = parse_csv_row(FIELDS, raw, SCHEMA)
    if errors:
        print(f"  {raw[0]:<8}: ✗ {errors}")
    else:
        print(f"  {raw[0]:<8}: ✓ {parsed}")

### Exercise 3 — Retry Logic

Write `retry(fn, args=(), max_attempts=3, exceptions=(Exception,))` that:
- Calls `fn(*args)` up to `max_attempts` times
- If it raises one of the listed `exceptions`, waits and tries again
- Returns the result on first success
- Raises the last exception if all attempts fail
- Prints which attempt is running

Test it with a function that randomly fails.

In [ ]:
import random

def retry(fn, args=(), max_attempts=3, exceptions=(Exception,)):
    """
    Call fn(*args) up to max_attempts times, retrying on specified exceptions.
    Returns the result on first success, raises the last exception if all fail.
    """
    # Your code here
    pass


# A flaky function that fails 60% of the time
def flaky_fetch(url):
    if random.random() < 0.6:
        raise ConnectionError(f"Timeout connecting to {url}")
    return {"status": 200, "data": [1, 2, 3]}

random.seed(42)
try:
    result = retry(flaky_fetch, args=("api.example.com",),
                   max_attempts=5, exceptions=(ConnectionError,))
    print(f"\nSuccess: {result}")
except ConnectionError as e:
    print(f"\nAll attempts failed: {e}")

### Exercise 4 — Exception-Aware Statistics

Write `robust_stats(values)` that computes mean, median, and std dev — but gracefully handles these edge cases using `try/except`:
- Empty list → return `{"error": "empty list"}`
- All values are None or non-numeric → return `{"error": "no valid values"}`
- Division by zero anywhere → catch and return `{"error": "calculation error"}`
- For valid data → return the full stats dict

In [ ]:
def robust_stats(values):
    """
    Compute mean, median, std — handling edge cases with try/except.
    """
    # Your code here
    pass


test_inputs = [
    ([85, 72, 91, 60, 38],              "normal data"),
    ([],                                "empty list"),
    ([None, None, None],               "all None"),
    (["abc", "xyz"],                    "all strings"),
    ([85, None, "bad", 72, None, 91],   "mixed valid/invalid"),
    ([42],                              "single value"),
]

for vals, label in test_inputs:
    result = robust_stats(vals)
    print(f"  {label:<30}: {result}")

### Exercise 5 — (Challenge) Custom Exception Hierarchy + Pipeline

Build a complete data validation system using custom exceptions:

1. Create: `DataError → MissingFieldError`, `DataError → ValidationError → RangeError`, `DataError → ValidationError → TypeMismatchError`

2. Write `validate_employee(record)` that raises the right exception for:
   - Missing `name` or `id` → `MissingFieldError`
   - Non-numeric `salary` → `TypeMismatchError`
   - Salary outside ₹10,000–₹5,00,000 → `RangeError`
   - Invalid `department` → `ValidationError`

3. Write `process_employees(records)` that catches each type separately, tracks counts per error type, and prints a final report.

In [ ]:
# Step 1: Define your exception hierarchy
class DataError(Exception): pass
# Your subclasses here


# Step 2: Write validate_employee
VALID_DEPTS = {"Engineering", "Marketing", "HR", "Finance", "Operations"}

def validate_employee(record):
    # Your code here
    pass


# Step 3: Write process_employees
def process_employees(records):
    # Your code here
    pass


# Test data
employees = [
    {"id": 1, "name": "Priya",  "salary": 85000,   "department": "Engineering"},
    {"id": 2, "name": "Rohan",  "salary": "abc",   "department": "Marketing"},    # bad salary
    {"id": 3, "name": "",       "salary": 62000,   "department": "HR"},           # empty name
    {"id": 4, "name": "Karan",  "salary": 5000,    "department": "Finance"},      # salary too low
    {"id": 5, "name": "Ananya", "salary": 70000,   "department": "Cooking"},      # bad dept
    {"id": 6, "name": "Dev",    "salary": 55000,   "department": "Operations"},
    {"id": 7},                                                                     # missing fields
]

process_employees(employees)

---
## 📌 Key Takeaways

- **Catch specific exceptions, not `Exception` or bare `except`.** Catching too broadly hides bugs in your own code. If a `NameError` or `AttributeError` slips through because you used `except Exception`, you'll spend hours wondering why your function returns `None` on data that looks fine.

- **Collect errors, don't abort.** In data pipelines, a single bad row should never kill the whole run. Use the `(results, error_log)` return pattern: process everything you can, record what failed and why, and return both. This is how production ETL systems work.

- **`finally` is for cleanup, `else` is for success logic.** If you open a file, database connection, or network socket in `try`, close it in `finally` — it runs whether or not an exception occurred. Use `else` (not the last line of `try`) for code that should only run when everything succeeded — it keeps the "happy path" visually separate from error handling.

---

## What's Next?

**Lesson 08 — File Handling (CSV, TXT)**  
You now have all the Python primitives to build real programs. The last piece: reading from and writing to files. You'll load a CSV dataset entirely with built-in Python — no Pandas — and experience firsthand why Pandas was invented. That contrast makes Module 5 much more meaningful.